# World Bank Table Metadata Setup

This notebook extracts table metadata (names and descriptions) from the `worldbank` schema 
and creates a searchable table that can be indexed by Vector Search.

**Output**: `main_catalog.dev.worldbank_tables` - a table with table names and descriptions

**Next step**: Run `vector_search_setup.ipynb` to create the Vector Search index and SQL function.

In [ ]:
# Configuration
CATALOG = "main_catalog"
SOURCE_SCHEMA = "worldbank"  # Schema containing the World Bank data tables
TARGET_SCHEMA = "dev"  # Schema where the search table will be created
TABLE_NAME = "worldbank_tables"

In [ ]:
from pyspark.sql.functions import col, when, concat, lit, udf
from pyspark.sql.types import StringType

---

## Step 1: Extract Table Metadata from Information Schema

In [ ]:
# Query information_schema to get all tables and their descriptions
tables_df = spark.sql(f"""
    SELECT 
        table_name,
        table_catalog,
        table_schema,
        comment as description,
        created as created_at,
        last_altered as updated_at
    FROM {CATALOG}.information_schema.tables
    WHERE table_schema = '{SOURCE_SCHEMA}'
      AND table_type = 'MANAGED'
    ORDER BY table_name
""")

print(f"Found {tables_df.count()} tables in {CATALOG}.{SOURCE_SCHEMA}")
display(tables_df.limit(10))

In [ ]:
# Check how many tables have descriptions
with_desc = tables_df.filter("description IS NOT NULL AND description != ''").count()
without_desc = tables_df.filter("description IS NULL OR description = ''").count()

print(f"Tables with descriptions: {with_desc}")
print(f"Tables without descriptions: {without_desc}")

---

## Step 2: Create Searchable Table with Embedding Text

In [ ]:
def table_name_to_indicator_name(table_name: str) -> str:
    """Convert sanitized table name to a readable indicator name.
    
    e.g., 'sp_pop_totl' -> 'SP.POP.TOTL'
    """
    return table_name.upper().replace('_', '.')

# Create UDF for indicator name conversion
indicator_name_udf = udf(table_name_to_indicator_name, StringType())

In [ ]:
# Build the searchable table with embedding text
search_table_df = tables_df.select(
    col("table_name"),
    indicator_name_udf(col("table_name")).alias("indicator_id"),
    concat(lit(f"{CATALOG}.{SOURCE_SCHEMA}."), col("table_name")).alias("full_table_name"),
    col("description"),
    col("created_at"),
    col("updated_at"),
    # Create embedding text: combine indicator ID and description
    when(
        col("description").isNotNull() & (col("description") != ""),
        concat(
            indicator_name_udf(col("table_name")),
            lit(": "),
            col("description")
        )
    ).otherwise(
        indicator_name_udf(col("table_name"))
    ).alias("embedding_text")
)

print("Sample of search table:")
display(search_table_df.limit(5))

In [ ]:
# Write to Delta table
full_table_name = f"{CATALOG}.{TARGET_SCHEMA}.{TABLE_NAME}"

search_table_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("delta.enableChangeDataFeed", "true") \
    .saveAsTable(full_table_name)

# Set table comment
spark.sql(f"""
    COMMENT ON TABLE {full_table_name} IS 
    'Searchable index of World Bank data tables. Use search_worldbank_tables() to find the right table for a given analysis need.'
""")

print(f"Created table: {full_table_name}")
print(f"Rows: {spark.table(full_table_name).count()}")

In [ ]:
# Verify the table
display(spark.sql(f"SELECT * FROM {full_table_name} LIMIT 10"))

---

## Next Steps

Run `vector_search_setup.ipynb` to:
1. Create the Vector Search index on this table
2. Create the `search_worldbank_tables()` SQL function

Example usage after setup:
```sql
SELECT * FROM main_catalog.dev.search_worldbank_tables('GDP growth')
```

---

## Refresh Table Metadata

Run this section to refresh the search table with latest table descriptions.

In [ ]:
# Set to True to refresh the table with latest metadata
REFRESH_TABLE = False

if REFRESH_TABLE:
    # Re-query information_schema
    tables_df = spark.sql(f"""
        SELECT 
            table_name,
            table_catalog,
            table_schema,
            comment as description,
            created as created_at,
            last_altered as updated_at
        FROM {CATALOG}.information_schema.tables
        WHERE table_schema = '{SOURCE_SCHEMA}'
          AND table_type = 'MANAGED'
        ORDER BY table_name
    """)
    
    # Rebuild search table
    search_table_df = tables_df.select(
        col("table_name"),
        indicator_name_udf(col("table_name")).alias("indicator_id"),
        concat(lit(f"{CATALOG}.{SOURCE_SCHEMA}."), col("table_name")).alias("full_table_name"),
        col("description"),
        col("created_at"),
        col("updated_at"),
        when(
            col("description").isNotNull() & (col("description") != ""),
            concat(
                indicator_name_udf(col("table_name")),
                lit(": "),
                col("description")
            )
        ).otherwise(
            indicator_name_udf(col("table_name"))
        ).alias("embedding_text")
    )
    
    # Overwrite table
    search_table_df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(full_table_name)
    
    print(f"Refreshed {full_table_name} with {search_table_df.count()} tables")
    print("\nRemember to sync the vector search index after refreshing!")
else:
    print("REFRESH_TABLE is False - no refresh performed.")
    print("Set REFRESH_TABLE = True and re-run to refresh.")